In [ ]:
""" Preliminaries, for developing and debugging in a python notebook.
    None of this is strictly necessary, but is useful for development.
"""

# Set up the path so our code is accessible w/o installing from pypi.
import sys, os
if os.path.abspath("../src") not in sys.path:
    sys.path.insert(0, os.path.abspath("../src"))

# Force automatic re-load of libraries on every call
# (useful for accelerated debugging of changing library code)
%load_ext autoreload
%autoreload 2

print(f"Executing notebook from '{os.getcwd()}'")
print("System paths available:")
for p in sys.path:
    print(f"  {p}")


In [ ]:
""" Define parameters for comparison

    Instructions:

    Set the five variables in this block, then run the entire notebook
    on completed matlab and python runs of STARE. The outputs should
    provide detailed comparisons of important stages of processing.

    The goal at the end of this block is to know where to find the python
    implementation's output (py_base_path / subject_name) and the matlab
    implementation's output (matlab_path / matlab_subject) for the same subject
    (implying identical input data). The output from this notebook will be
    written to comparison_path.
    
    Warning!!!
    This will only work on python STARE output later than 1/31/2024 because
    I've changed the definitions of the Centroid and TimeActivityCurve classes.
    The old binary objects are not compatible with the new binary objects.
    
"""

from pathlib import Path
from starelib.timeactivitycurve import TimeActivityCurve


subject_name = "NHFDG003"
matlab_subject = f"20230612_{subject_name}"

# After I ran STARE, I moved all the data to a server with a much bigger disk
# than my laptop. This points to where I've mounted it locally.
stare_output_path = Path("/home/mike/mnt/lorenzdata/pet/derivatives/from_henry")

# My matlab-based STARE runs saved subjects into this directory:
matlab_path = stare_output_path / "test_matlab"
matlab_debug_path = matlab_path / matlab_subject / "debug"

# My python-based STARE runs saved some comparable data into this directory.
py_base_path = stare_output_path / "benchmarking_py063_OMP-6_balanced_vc0"

# To support sharing with others, I saved output from this process into
# a separate location, within my Dropbox path
dropbox_path = Path("/home/mike/Dropbox")
comparison_path = dropbox_path / "Projects" / "stare" / "comparisons_20230710"


## Load Matlab and Python data

In [ ]:
""" Load matlab TAC data """

import pandas as pd
import numpy as np


# Load matlab data for visualization. I modified matlab STARE to save out
# csv files of all these data.
ml_mid_times = pd.read_csv(
    matlab_debug_path / f"{subject_name}_midtimes.csv",
    sep=',', header=None, index_col=None
)[0].values
ml_kmeans_1_vascular_tac = pd.read_csv(
    matlab_debug_path / f"{subject_name}_step_1_vascular_tac.csv",
    sep=',', header=None, index_col=None
)[0].values
ml_kmeans_2_vascular_tac = pd.read_csv(
    matlab_debug_path / f"{subject_name}_step_2_vascular_tac.csv",
    sep=',', header=None, index_col=None
)[0].values
ml_pvc_mean_vascular_tac = pd.read_csv(
    matlab_debug_path / f"{subject_name}_pvc_mean_vascular_tac.csv",
    sep=',', header=None, index_col=None
)[0].values
ml_pvc_sd_vascular_tac = pd.read_csv(
    matlab_debug_path / f"{subject_name}_pvc_sd_vascular_tac.csv",
    sep=',', header=None, index_col=None
)[0].values

# After loading the data, I create TimeActivityCurve objects, as defined
# in python STARE, and imported in the block above,
# to organize the activity over time. These objects are also readable by my
# plotting functions.
matlab_tacs = {
    "kmeans_step_1": TimeActivityCurve(
        activity=ml_kmeans_1_vascular_tac, timepoints=ml_mid_times,
        source="matlab", name="matlab kmeans 1",
    ),
    "kmeans_step_2": TimeActivityCurve(
        activity=ml_kmeans_2_vascular_tac, timepoints=ml_mid_times,
        source="matlab", name="matlab kmeans 2",
    ),
    "pvc_vascular": TimeActivityCurve(
        activity=ml_pvc_mean_vascular_tac, timepoints=ml_mid_times,
        sd=ml_pvc_sd_vascular_tac,
        source="matlab", name="matlab pvc",
    ),
}


In [ ]:
""" Load matlab randomized curve bootstraps """

ml_boot_curve_permutations = pd.read_csv(
    matlab_debug_path / f"boot_curve_permutations.csv",
    sep=',', header=None, index_col=None
).values


In [ ]:
""" Load python data """

import pickle


python_path = py_base_path
python_debug_path = python_path / subject_name / "debug"
python_cache_path = python_path / subject_name / "cache"

# Load python TACs
# Here, we unpickle the binary pickled Centroid objects
python_tacs = {
    # "plasma": pickle.load(
    #     open(python_debug_path / f"sub-{subject_name}_tac_plasma.pkl", "rb")
    # ),
    "kmeans_step_1": pickle.load(
        open(python_debug_path / f"sub-{subject_name}_step-1-1_kmeans_centroid.pkl", "rb")
    ),
    "kmeans_step_2": pickle.load(
        open(python_debug_path / f"sub-{subject_name}_step-1-2_kmeans_centroid.pkl", "rb")
    ),
    "pvc_vascular": pickle.load(
        open(python_debug_path / f"sub-{subject_name}_tac_pvc.pkl", "rb")
    ),
}

# Manual editing to clean up some unintended naming
python_tacs["kmeans_step_1"].name = "python kmeans 1"
python_tacs["kmeans_step_2"].name = "python kmeans 2"
python_tacs["pvc_vascular"].name = "python pvc"

# The pickled objects are custom classes, defined in the python STARE code.
print("python_tacs is a dict (", type(python_tacs), "), and its values:")
for k, v in python_tacs.items():
    print(f"  key '{k}' is type '{type(v)}'")


In [ ]:
""" Load python randomized curve bootstraps """

# This pickle contains a list of 500 bootstrap curves,
# and each curve is a dict containing 7 features, one of which is a
# 22-item vector called 'bootstrap_curve'.
python_boot_curves_list = pickle.load(
    open(python_debug_path / f"sub-{subject_name}_boot_anchor_data.pkl", "rb")
)['curve_fits']

# This list comprehension extracts the 'bootstrap_curve' vector from each
# dict and vertically stacks it into an array. Because of the vertical stacking,
# the array is [500 rows x 22 columns], but it's transposed by the .T at the
# end, making the final 'python_boot_curves' a [22 row x 500 column] array.
python_boot_curves = np.vstack([
    _['bootstrap_curve'] for _ in python_boot_curves_list
]).T


## Plot TACs

In [ ]:
import seaborn as sns


# Combine all TACs from matlab and python into one list.
# The TAC attributes should allow discrimination in the plotter.
all_tacs = (
    [v for k, v in matlab_tacs.items()] +
    [v for k, v in python_tacs.items()]
)
# all_tacs extracts 'kmeans_step_1', 'kmeans_step_2', and 'pvc_vascular'
# from each result, giving us 6 TACs we can plot. Each TAC here is packaged
# into a Centroid or TimeActivityCurve object, which is interpretable by the
# plotting function.


In [ ]:
# Here, we use the 'plot_detailed_tacs' function from within STARE.
# It will build the figure and return a matplotlib figure object.
# You can manipulate the colors and line styles via the 'palette' and 'dashes'
# dicts sent into the function.
from starelib.plotting import plot_detailed_tacs

# Generate a plot of all TACs in the same space.
sns.set_style('whitegrid')
fig_vasc_tacs = plot_detailed_tacs(
    data=all_tacs,
    title=f"Subject {subject_name} TACs",
    palette={
        "matlab kmeans": "cornflowerblue",
        "matlab kmeans 1": "cornflowerblue",
        "matlab kmeans 2": "cornflowerblue",
        "matlab pvc": "blue",
        "python kmeans": "limegreen",
        "python kmeans 1": "limegreen",
        "python kmeans 2": "limegreen",
        "python pvc": "green",
        "plasma": "darkred",
    },
    dashes={
        "matlab kmeans": (1, 1),
        "matlab kmeans 1": (1, 2),
        "matlab kmeans 2": (2, 1),
        "matlab pvc": "",
        "python kmeans": (1, 1),
        "python kmeans 1": (1, 2),
        "python kmeans 2": (2, 1),
        "python pvc": "",
        "plasma": "",
    },
)


In [ ]:
# Now we make sure our output path exists, making it and a subdirectory if not.
comparison_path.mkdir(exist_ok=True)
(comparison_path / subject_name).mkdir(exist_ok=True)

# And we save the figure to disk as a png image.
fig_vasc_tacs.savefig(
    comparison_path / subject_name / f"{subject_name}_vascular_tacs.png"
)


## Plot boot curves

In [ ]:
""" Load boot curve data from both sources. """

# MATLAB boot curves
ml_boot_curve_df = pd.DataFrame(
    data=ml_boot_curve_permutations
)

# Python boot curves
py_boot_curve_df = pd.DataFrame(
    data=python_boot_curves
)


In [ ]:
""" Plot the boot curve data """

from starelib.plotting import plot_many_curves

fig_bootstraps = plot_many_curves(
    py_boot_curve_df, python_tacs['pvc_vascular'],
    ml_boot_curve_df, matlab_tacs['pvc_vascular'],
    subject_name,
)
fig_bootstraps.savefig(comparison_path / subject_name / f"{subject_name}_bootstrap_curves.png")


This is just some manually saved information from prior runs I've used
for comparison and debugging as I made code changes.

Old values at the peak, before fixing SD problem:
(1.8023, 2.9741, 4.1605)

Next values at the peak, with good SD but wrong (fit rather than pvc) TAC:
(0.6256, 2.9693, 5.3420)

In [ ]:
""" Current, correct, values at the peak """
print("({:0.4f}, {:0.4f}, {:0.4f})".format(
    python_tacs['pvc_vascular'].activity[4] - python_tacs['pvc_vascular'].sd[4],
    python_tacs['pvc_vascular'].activity[4],
    python_tacs['pvc_vascular'].activity[4] + python_tacs['pvc_vascular'].sd[4],
))

----

## And now for the k constants

In [ ]:
""" Load and format matlab-discovered constants """

ml_rate_constants = pd.read_csv(
    matlab_debug_path / "boot_rate_constants.csv",
    sep=',', header=None, index_col=None,
).values
ml_rate_constants = ml_rate_constants[~np.isnan(ml_rate_constants).any(axis=1)]
ml_ki_constants = pd.read_csv(
    matlab_debug_path / "boot_kis.csv",
    sep=',', header=None, index_col=None,
).values
ml_ki_constants = ml_ki_constants[~np.isnan(ml_ki_constants).any(axis=1)]


In [ ]:
""" Load and format python-discovered constants. """

py_stuff = pickle.load(
    open(python_debug_path / f"sub-{subject_name}_boot_anchor_data.pkl", "rb")
)
py_rate_constants = py_stuff['good_rate_constants']
py_ki_constants = py_stuff['kis']
region_names = py_stuff['regional_tacs'].columns


In [ ]:
from starelib.plotting import plot_regional_densities


constants = ["k1", "k2", "k3", ]
figures = {}

for i, k in enumerate(constants):
    # Build indices to take only matlab columns relevant to this constant.
    matlab_k_indices = np.array([
        _ for _ in range(len(region_names) * len(constants))
        if (i * len(region_names)) <= _ < ((i + 1) * len(region_names))
    ])
    figures[k] = plot_regional_densities(
        region_names=region_names,
        rate_constants=py_rate_constants[:, :, i],
        hist_color='palegreen', fwhm_color='green',
        comp_rate_constants=np.take(ml_rate_constants,
                                    matlab_k_indices, axis=1),
        comp_hist_color='cornflowerblue', comp_fwhm_color='blue',
        num_bootstraps=1000,
        coefficient=k,
        subject=subject_name,
        verbose=False
    )
    figures[k].savefig(comparison_path / subject_name / f"{subject_name}_rate_constant_kernel_density_{k}.png")


In [ ]:
figure_ki = plot_regional_densities(
    region_names=region_names,
    rate_constants=py_ki_constants,
    hist_color='palegreen', fwhm_color='green',
    comp_rate_constants=ml_ki_constants,
    comp_hist_color='cornflowerblue', comp_fwhm_color='blue',
    num_bootstraps=1000,
    coefficient="ki",
    subject=subject_name,
    verbose=False
)
figure_ki.savefig(comparison_path / subject_name / f"{subject_name}_rate_constant_kernel_density_ki.png")


----

## And plot the final fitted curves from python and matlab


In [ ]:
""" Load the fitted curve data. """

bootstrap_curve_fits = pickle.load(
    open(python_cache_path / f"sub-{subject_name}_step-4_bootstrap_curve_fits.pkl", "rb")
)
py_fitted_curves_df = pd.DataFrame(
    [_['fit_exp']['curve'] for _ in bootstrap_curve_fits]
).T
py_fitted_tac = py_stuff['uniform_tac']

ml_fitted_curves_matrix = pd.read_csv(
    matlab_debug_path / "boot_curves_fit.csv",
    sep=',', header=None, index_col=None,
).values
ml_uniform_timepoints = ml_fitted_curves_matrix[:, 0]
ml_uniform_activity = ml_fitted_curves_matrix[:, 1:]
ml_uniform_activity = ml_uniform_activity[:, ~np.isnan(ml_uniform_activity).any(axis=0)]
ml_fitted_curves_df = pd.DataFrame(data=ml_uniform_activity)

ml_uniform_fitted_tac = pd.read_csv(
    matlab_debug_path / "uniform_fit_vascular_tac.csv",
    sep=',', header=None, index_col=None,
)[0].values
ml_uniform_fit_vascular_tac = TimeActivityCurve(
    activity=ml_uniform_fitted_tac,
    timepoints=ml_uniform_timepoints,
    source='matlab',
)


In [ ]:
print(py_fitted_curves_df.shape, ml_fitted_curves_df.shape)
print(len(py_fitted_tac.activity), len(py_fitted_tac.timepoints))
print(len(ml_uniform_fit_vascular_tac.activity), len(ml_uniform_fit_vascular_tac.timepoints))


In [ ]:
""" Plot the fitted boot curve data """

from starelib.plotting import plot_many_curves

ml_fitted_curves_df = pd.DataFrame(np.zeros((551, 1)))
fig_fitted_bootstraps = plot_many_curves(
    py_fitted_curves_df, py_fitted_tac,
    ml_fitted_curves_df, ml_uniform_fit_vascular_tac,
    subject_name, skip_outliers=True
)
fig_fitted_bootstraps.savefig(comparison_path / subject_name / f"{subject_name}_fitted_bootstrap_curves.png")


In [ ]:
py_sa_bounds, py_annealer_results, py_final_rate_df = pickle.load(
    open(python_cache_path / f"sub-{subject_name}_step-5_minimized_params.pkl", "rb")
)


In [ ]:
from starelib.plotting import plot_detailed_tacs

plot_detailed_tacs([_['hires_tac'] for _ in py_annealer_results], title="Hi-Res TACs")
plot_detailed_tacs([_['source_tac'] for _ in py_annealer_results], title="Source TACs")
